# Egyptian Real Estate EDA

### setup

In [57]:
import os
import sys

# 1. Set the IP to localhost to avoid the "Invalid Spark URL" error
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 2. Point to your Spark and Hadoop folders
os.environ['SPARK_HOME'] = r'C:\spark-4.1.1-bin-hadoop3\spark-4.1.1-bin-hadoop3'
os.environ['HADOOP_HOME'] = r'C:\Hadoop' 
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Add PySpark to your sys.path
sys.path.append(os.path.join(os.environ['SPARK_HOME'], 'python'))
sys.path.append(os.path.join(os.environ['SPARK_HOME'], 'python', 'lib', 'py4j-0.10.9.9-src.zip'))

# 4. Initialize Spark
from pyspark import SparkConf
from pyspark.sql import SparkSession
conf = SparkConf()
conf.set("spark.python.worker.timeout", "1200") # Set to 20 minutes
conf.set("spark.driver.memory", "4g")
conf.set("spark.executor.memory", "2g")
# If using a local-mode or single-node K8s, sometimes this helps:
conf.set("spark.driver.bindAddress", "127.0.0.1")
spark = SparkSession.builder \
    .appName("JupyterSparkTest") \
    .master("local[*]") \
    .config(conf=conf) \
    .getOrCreate()

sc = spark.sparkContext

### Load And Inspect Dataset

In [35]:
import pandas as pd
import numpy as np

In [115]:
# Define the path to your Egypt Property Prices dataset
PATH = "data/raw/propertyFinder.csv"

# Load the data into a Spark DataFrame
# inferSchema=True: Automatically detects data types (integers, strings, etc.)
df_raw = spark.read.csv(PATH, header=True, inferSchema=True)

# Print basic info to verify ingestion (Project Pipeline Step II)
print(f"Total Records: {df_raw.count():,}")
print(f"Total Columns: {len(df_raw.columns)}")
df_raw.printSchema()
df_raw.show(5)

Total Records: 39,713
Total Columns: 53
root
 |-- listing_id: string (nullable = true)
 |-- internal_id: double (nullable = true)
 |-- category: string (nullable = true)
 |-- listing_type: string (nullable = true)
 |-- detail_url: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- offering_type: string (nullable = true)
 |-- completion_status: string (nullable = true)
 |-- title: string (nullable = true)
 |-- price_egp: string (nullable = true)
 |-- price_period: string (nullable = true)
 |-- price_currency: string (nullable = true)
 |-- location_full: string (nullable = true)
 |-- city: string (nullable = true)
 |-- town: string (nullable = true)
 |-- district: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- area_value: double (nullable = true)
 |-- area_unit: string (nullable = true)
 |-

### Missing counts 

In [116]:
from pyspark.sql.functions import col, count, when
import pandas as pd
import builtins

# Calculate missing counts for every column using Spark
total_rows = df_raw.count()
missing_data = []

missing_counts = df_raw.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_raw.columns
])

counts_row = missing_counts.collect()[0]

for column in df_raw.columns:
    count_val = int(counts_row[column])
    if count_val > 0:
        percentage = builtins.round((count_val / total_rows) * 100, 1)
        missing_data.append((column, count_val, percentage))

# Create Pandas DataFrame and sort by missing count
result_df = pd.DataFrame(missing_data, columns=["Column", "Missing Count", "Percentage"])
result_df = result_df.sort_values("Missing Count", ascending=False).reset_index(drop=True)

print(result_df)

               Column  Missing Count  Percentage
0                rera          39707       100.0
1           video_url          39504        99.5
2     agent_languages          29669        74.7
3   completion_status          19950        50.2
4      payment_method          19200        48.3
5           furnished           8384        21.1
6         subdistrict           6756        17.0
7        broker_phone            426         1.1
8            district            347         0.9
9        broker_email            308         0.8
10     agent_is_super            219         0.6
11      contact_phone            106         0.3
12          broker_id             96         0.2
13        broker_name             94         0.2
14        agent_email             76         0.2
15   contact_whatsapp             67         0.2
16      contact_email             67         0.2
17         agent_name             48         0.1
18         scraped_at             44         0.1
19          amenitie

## Data Cleaning 

### Drop columns that are not relevant 

In [117]:
drop_cols = [
    "internal_id", "detail_url", "listing_type", "reference",
    "price_currency", "area_unit", "offering_type",
    "agent_id", "agent_name", "agent_email", "agent_is_super", "agent_languages",
    "broker_id", "broker_name", "broker_email", "broker_phone",
    "contact_phone", "contact_whatsapp", "contact_email",
    "scraped_at", "listed_date_parsed", "title", "description", "video_url", "rera"
]

# Drop columns that exist in the DataFrame
cols_to_drop = [c for c in drop_cols if c in df_raw.columns]
df = df_raw.drop(*cols_to_drop)

print(f"Columns before drop: {len(df_raw.columns)}")
print(f"Columns after drop: {len(df.columns)}")
print(f"Dropped columns: {len(cols_to_drop)}")

Columns before drop: 53
Columns after drop: 29
Dropped columns: 24


### removing duplicates

In [118]:
# Filter to buy listings only
df = df.filter(col("category") == "buy")

# Remove duplicates based on listing_id, keeping first occurrence
df = df.dropDuplicates(subset=["listing_id"])

# Count buy listings after deduplication
buy_count = df.count()
print(f"Buy listings after dedup: {buy_count:,}")

Buy listings after dedup: 19,803


### Type handling and removing nulls 

In [119]:
from pyspark.sql.functions import col, when, coalesce, lit, regexp_replace, lower, trim
from pyspark.sql.types import DoubleType

initial_count = df.count()
print(f"Initial row count: {initial_count:,}")

# Regex to check if a string is a valid numeric format (integer or decimal)
# This prevents Spark from attempting to cast garbage like "7+" or "."
is_numeric_regex = "^[0-9]+(\\.[0-9]*)?$"

# 1. Handle bedrooms: Studio -> 1.0, 7+ -> 7.0, others check regex then cast
df = df.withColumn("bedrooms",
    when(lower(trim(col("bedrooms"))) == "studio", lit(1.0))
    .when(trim(col("bedrooms")).rlike("^7\\+$"), lit(7.0))
    .when(trim(col("bedrooms")).rlike(is_numeric_regex), col("bedrooms").cast(DoubleType()))
    .otherwise(None)
)

# 2. Handle bathrooms: 7+ -> 7.0, others check regex then cast
df = df.withColumn("bathrooms",
    when(trim(col("bathrooms")).rlike("^7\\+$"), lit(7.0))
    .when(trim(col("bathrooms")).rlike(is_numeric_regex), col("bathrooms").cast(DoubleType()))
    .otherwise(None)
)

# 3. Clean numeric columns: price_egp, area_value, images_count
numeric_cols = ["price_egp", "area_value", "images_count"]
for col_name in numeric_cols:
    # Strip non-numeric except dot
    cleaned_expr = regexp_replace(col(col_name), "[^0-9.]", "")
    
    df = df.withColumn(col_name,
        when(
            cleaned_expr.rlike(is_numeric_regex),
            cleaned_expr.cast(DoubleType())
        ).otherwise(None)
    )

# 4. Calculate medians (Your exact logic)
df_numeric = df.filter(col("bedrooms").isNotNull() & col("bathrooms").isNotNull())
bedroom_medians = df_numeric.approxQuantile("bedrooms", [0.5], 0.01)
bathroom_medians = df_numeric.approxQuantile("bathrooms", [0.5], 0.01)

bedroom_median = bedroom_medians[0] if bedroom_medians else 2.0
bathroom_median = bathroom_medians[0] if bathroom_medians else 1.0

# Count rows with nulls that will be imputed (modified)
rows_with_null_bedroom = df.filter(col("bedrooms").isNull()).count()
rows_with_null_bathroom = df.filter(col("bathrooms").isNull()).count()
rows_with_null_furnished = df.filter(col("furnished").isNull()).count()

# 5. Fill nulls and handle categorical (Your exact logic)
df = df.withColumn("bedrooms", coalesce(col("bedrooms"), lit(bedroom_median)))
df = df.withColumn("bathrooms", coalesce(col("bathrooms"), lit(bathroom_median)))
df = df.withColumn("images_count", coalesce(col("images_count"), lit(0.0)))
df = df.withColumn("furnished", coalesce(col("furnished"), lit("NO")))
df = df.withColumn("completion_status", coalesce(col("completion_status"), lit("unknown")))
df = df.withColumn("payment_method", coalesce(col("payment_method"), lit("unknown")))

# 6. Drop critical nulls and zeros
count_before_null_drop = df.count()
df = df.dropna(subset=["price_egp", "area_value", "lat", "lon"])
df = df.filter((col("price_egp") > 0) & (col("area_value") > 0))

null_drop_count = count_before_null_drop - df.count()
final_count = df.count()

# Calculate total rows modified (nulls filled or values corrected)
total_modified = rows_with_null_bedroom + rows_with_null_bathroom + rows_with_null_furnished

# Summary Output
print("-" * 55)
print(f"DATA CLEANING REPORT")
print("-" * 55)
print(f"Initial rows:                                  {initial_count:,}")
print(f"Rows modified (nulls imputed):                 {total_modified:,}")
print(f"Rows dropped due to critical nulls/invalid:    {null_drop_count:,}")
print(f"Final clean rows:                              {final_count:,}")
print("-" * 55)
print(f"Bedroom median (imputed):                      {bedroom_median:.1f}")
print(f"Bathroom median (imputed):                     {bathroom_median:.1f}")

Initial row count: 19,803
-------------------------------------------------------
DATA CLEANING REPORT
-------------------------------------------------------
Initial rows:                                  19,803
Rows modified (nulls imputed):                 8,304
Rows dropped due to critical nulls/invalid:    6
Final clean rows:                              19,797
-------------------------------------------------------
Bedroom median (imputed):                      3.0
Bathroom median (imputed):                     3.0


### Standardization of categorial columns 

In [120]:
from pyspark.sql.functions import col, lower, trim, when

# 1. Capture count before modification
before_payment_filter = df.count()

# 2. Standardize the values and mark 'others' as None
df = df.withColumn("payment_method", 
    when(lower(trim(col("payment_method"))) == "cash", "cash")
    .when(lower(trim(col("payment_method"))) == "installments", "installments")
    .when(lower(trim(col("payment_method"))) == "cash | installments", "both")
    .otherwise(None)
)

# 3. Drop the 'others' (which are now null)
df = df.dropna(subset=["payment_method"])

# 4. Logging the changes
after_payment_filter = df.count()
dropped_payment = before_payment_filter - after_payment_filter

print("-" * 30)
print(f"PAYMENT METHOD CLEANING")
print("-" * 30)
print(f"Rows dropped (other payment methods): {dropped_payment:,}")
print(f"Final usable rows:                   {after_payment_filter:,}")
print("-" * 30)

# 5. Verify the unique values left
print("Remaining categories in payment_method:")
df.select("payment_method").distinct().show()

------------------------------
PAYMENT METHOD CLEANING
------------------------------
Rows dropped (other payment methods): 485
Final usable rows:                   19,312
------------------------------
Remaining categories in payment_method:
+--------------+
|payment_method|
+--------------+
|          both|
|          cash|
|  installments|
+--------------+



### Outlier Removal

In [ ]:
from pyspark.sql.functions import col, min as spark_min, max as spark_max

# 1. Create the price_per_sqm column
df = df.withColumn("price_per_sqm", col("price_egp") / col("area_value"))

# 2. Calculate the 1st and 99th percentiles
# We use a relative error of 0.01 for a good balance of speed and accuracy
price_quantiles = df.approxQuantile("price_per_sqm", [0.01, 0.99], 0.01)
lo_price = price_quantiles[0]
hi_price = price_quantiles[1]

# Set area bounds
lo_area = 20
hi_area = 1500

# 3. Capture count before filtering
count_before_outliers = df.count()

# 4. Apply the outlier filters
df = df.filter(
    (col("price_per_sqm") >= lo_price) & (col("price_per_sqm") <= hi_price) &
    (col("area_value") >= lo_area) & (col("area_value") <= hi_area)
)

# 5. Capture count after filtering
count_after_outliers = df.count()
outliers_removed = count_before_outliers - count_after_outliers

# 6. Get final min/max for the log
stats = df.select(
    spark_min("price_per_sqm").alias("min_sqm"),
    spark_max("price_per_sqm").alias("max_sqm")
).collect()[0]

# --- Final Log Output ---
print("-" * 50)
print(f"OUTLIER REMOVAL REPORT")
print("-" * 50)
print(f"Removed {outliers_removed:,} outliers")
print(f"Final clean dataset: {count_after_outliers:,} rows")
print(f"Price/sqm range: {stats['min_sqm']:,.0f} - {stats['max_sqm']:,.0f} EGP")
print("-" * 50)

--------------------------------------------------
OUTLIER REMOVAL REPORT
--------------------------------------------------
Removed 34 outliers
Final clean dataset: 19,148 rows
Price/sqm range: 558 - 4,710,145 EGP
--------------------------------------------------


### Cleaned Data Overview

In [124]:
from pyspark.sql.functions import count, when, col

# 1. Get Shape (Rows, Columns)
rows = df.count()
cols = len(df.columns)

# 2. Calculate Total Missing Values (Safe for all data types)
# We use isNull() which works for Strings, Numbers, and Dates.
missing_values_expr = [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
missing_counts_row = df.select(missing_values_expr).collect()[0]

# Sum up all the counts from the row
total_missing = sum([missing_counts_row[c] for c in df.columns])

# 3. Output basic info
print(f"Shape: ({rows}, {cols})")
print(f"Missing values: {total_missing}")

# 4. Get Descriptive Stats for Price per SQM
print(f"\nPrice per sqm stats:")
# Using summary() provides a clean table of the key metrics
df.select("price_per_sqm").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

Shape: (19148, 30)
Missing values: 4051

Price per sqm stats:
+-------+-----------------+
|summary|    price_per_sqm|
+-------+-----------------+
|  count|            19148|
|   mean|79326.61979231895|
| stddev| 88256.1607805236|
|    min|557.9399141630902|
|    25%|41481.48148148148|
|    50%|63829.78723404255|
|    75%|94871.79487179487|
|    max|4710144.927536231|
+-------+-----------------+



### Save Cleaned Data

In [128]:
import os

# 1. Define the output path
output_dir = "data/processed"
output_file = os.path.join(output_dir, "propertyFinder_cleaned.csv")

# 2. Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

# 3. Convert Spark DF to Pandas
# This brings the data into local memory as a single table
# Similar to your input 'propertyFinder.csv'
final_pandas_df = df.toPandas()

# 4. Save to CSV
# index=False: prevents an extra column of numbers at the start
# encoding='utf-8-sig': ensures Arabic names (cities/districts) show up correctly in Excel
final_pandas_df.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"Export Complete!")


Export Complete!
